# Michigan DNR Trail Data Audit

This notebook validates the downloaded Upper Peninsula hiking-trail segments, converts source placeholder values to missing values, and prepares a reduced GeoDataFrame for downstream trail grouping and filtering.

In [16]:
from pathlib import Path

import geopandas as gpd
import pandas as pd


DATA_PATH = Path("../data/raw/dnr_up_hiking_trails.geojson")
PROCESSED_PATH = Path("../data/processed/dnr_up_hiking_segments_clean.parquet")

trails = gpd.read_file(DATA_PATH)

print(f"Rows: {len(trails):,}")
print(f"Columns: {len(trails.columns)}")
print(f"CRS: {trails.crs}")
trails.head()

Rows: 2,439
Columns: 18
CRS: EPSG:4326


,OBJECTID,TrailNamePrimary,HikingName,FacilityName,County,Peninsula,Hiking,TrailApprovalStatus,OpenClosedStatusNonmotor,SurfaceType,ADAAccessible,SegmentLengthMiles,SpecialRestrictionType,TrailAdministrator,RecreationSearchFacilityID,RecreationSearchTrailID,last_edited_date,geometry
0,1919,Straits - Main Trail,Straits Main Trail,Straits State Park,Mackinac,Upper Peninsula,Hiking,Approved,Open,Dirt Natural,Not Accessible,0.006157,,DNR Parks And Recreation PRD,d5ed4bad-cb4e-46cc-b68c-a89e06307b1f,4609ed9c-e1c0-481d-a8bd-d4ed1639e6c0,1770930791000,"LINESTRING (-84.72138 45.84982, -84.72135 45.8..."
1,2486,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Upper Peninsula,Hiking,Approved,Temporarily Closed,Dirt Natural,Not Accessible,1.200800,-1,DNR Parks And Recreation PRD,3f6bcdb2-822b-49ae-84f7-7321edbb00e6,23531368-cb4b-4832-adad-4699cdb5e714,1770930791000,"LINESTRING (-87.47649 46.63472, -87.47644 46.6..."
2,2495,UP 2,North Country Trail,-1,Mackinac,Upper Peninsula,Hiking,Approved,Open,Dirt Natural,Not Accessible,3.891672,-1,DNR Parks And Recreation PRD,Unspecified,Unspecified,1780334188000,"LINESTRING (-84.73683 45.87386, -84.73687 45.8..."
3,2540,Gemini Lake Pathway,Gemini Lake Pathway,State Forest,Schoolcraft,Upper Peninsula,Hiking,Approved,Open,Dirt Natural,Not Accessible,0.514227,-1,DNR Parks And Recreation PRD,Unspecified,59407270-ea7e-4150-ad57-9cf5fcbd8397,1770930791000,"LINESTRING (-86.30655 46.48498, -86.30623 46.4..."
4,2541,Gemini Lake Pathway,Gemini Lake Pathway,State Forest,Schoolcraft,Upper Peninsula,Hiking,Approved,Open,Dirt Natural,Not Accessible,0.264824,-1,DNR Parks And Recreation PRD,Unspecified,59407270-ea7e-4150-ad57-9cf5fcbd8397,1770930791000,"LINESTRING (-86.30464 46.48159, -86.30522 46.4..."


## 1. Dataset overview

In [17]:
# Summarize field cardinality before applying cleaning rules.
unique_counts = (
    trails.nunique(dropna=False)
    .rename("unique_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

unique_counts

,column,unique_count
0,OBJECTID,2439
1,TrailNamePrimary,119
2,HikingName,135
3,FacilityName,25
4,County,15
5,Peninsula,1
6,Hiking,1
7,TrailApprovalStatus,1
8,OpenClosedStatusNonmotor,4
9,SurfaceType,11


## 2. Placeholder-value audit

In [ ]:
PLACEHOLDER_VALUES = {
    "",
    "Unspecified",
    "Unknown",
    "None",
    "N/A",
}

text_columns = trails.select_dtypes(
    include=["object", "string"]
).columns

placeholder_records = []

for column in text_columns:
    normalized = trails[column].astype("string").str.strip()
    matches = normalized.isin(PLACEHOLDER_VALUES)

    placeholder_records.append(
        {
            "column": column,
            "placeholder_count": int(matches.sum()),
            "placeholder_percent": round(matches.mean() * 100, 2),
            "values_found": sorted(
                normalized.loc[matches].dropna().unique().tolist()
            ),
        }
    )

placeholder_audit = (
    pd.DataFrame(placeholder_records)
    .sort_values("placeholder_count", ascending=False)
    .reset_index(drop=True)
)

placeholder_audit

,column,placeholder_count,placeholder_percent,values_found
0,SpecialRestrictionType,2389,97.95,"[, -1]"
1,RecreationSearchFacilityID,1021,41.86,[Unspecified]
2,RecreationSearchTrailID,867,35.55,[Unspecified]
3,FacilityName,658,26.98,"[-1, Unspecified]"
4,SurfaceType,13,0.53,[-1]
5,TrailAdministrator,5,0.21,[-1]
6,Hiking,0,0.00,[]
7,Peninsula,0,0.00,[]
8,County,0,0.00,[]
9,HikingName,0,0.00,[]


In [19]:
# Preserve the raw import and clean text fields on a separate copy.
audit_trails = trails.copy()

for column in text_columns:
    cleaned = audit_trails[column].astype("string").str.strip()
    audit_trails[column] = cleaned.mask(
        cleaned.isin(PLACEHOLDER_VALUES),
        pd.NA,
    )

In [20]:
logical_missing = (
    audit_trails.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

logical_missing["missing_percent"] = (
    logical_missing["missing_count"] / len(audit_trails) * 100
).round(2)

logical_missing.sort_values(
    "missing_count",
    ascending=False,
)

,missing_count,missing_percent
SpecialRestrictionType,2389,97.95
RecreationSearchFacilityID,1021,41.86
RecreationSearchTrailID,867,35.55
FacilityName,658,26.98
SurfaceType,13,0.53
TrailAdministrator,5,0.21
TrailNamePrimary,0,0.00
OBJECTID,0,0.00
County,0,0.00
HikingName,0,0.00


## 3. Candidate trail grouping

In [21]:
# Evaluate HikingName as a user-facing grouping field.
hiking_name_audit = (
    audit_trails.groupby("HikingName", dropna=False)
    .agg(
        segment_count=("OBJECTID", "size"),
        county_count=("County", "nunique"),
        facility_count=("FacilityName", "nunique"),
        primary_name_count=("TrailNamePrimary", "nunique"),
        counties=(
            "County",
            lambda values: sorted(
                values.dropna().astype(str).unique()
            ),
        ),
        facilities=(
            "FacilityName",
            lambda values: sorted(
                values.dropna().astype(str).unique()
            ),
        ),
    )
    .sort_values(
        ["county_count", "facility_count", "segment_count"],
        ascending=False,
    )
)

print(f"Unique hiking names: {len(hiking_name_audit):,}")
hiking_name_audit.head(30)

Unique hiking names: 135


,segment_count,county_count,facility_count,primary_name_count,counties,facilities
HikingName,,,,,,
North Country Trail,597,9,6,10,['Alger' 'Baraga' 'Chippewa' 'Gogebic' 'Hought...,['Craig Lake State Park' 'Porcupine Mountains ...
Iron Belle Trail,15,7,1,4,['Delta' 'Luce' 'Mackinac' 'Marquette' 'Menomi...,['Porcupine Mountains Wilderness State Park']
Porcupine Mountain Wilderness Lake Superior Trail,128,2,1,3,['Gogebic' 'Ontonagon'],['Porcupine Mountains Wilderness State Park']
Porcupine Mts Big Carp River Trail,78,2,1,2,['Gogebic' 'Ontonagon'],['Porcupine Mountains Wilderness State Park']
Porcupine Mountain Wilderness - Lake Superior Trail,72,2,1,1,['Gogebic' 'Ontonagon'],['Porcupine Mountains Wilderness State Park']
Fox River Pathway,24,2,1,2,['Alger' 'Schoolcraft'],['State Forest']
Tahquamenon Falls State Park Lower Falls Foot Trails,22,2,1,2,['Chippewa' 'Luce'],['Tahquamenon Falls State Park']
Tahquamenon Falls River Trail,14,2,1,1,['Chippewa' 'Luce'],['Tahquamenon Falls State Park']
Porcupine Mountain Wilderness Cross Trail Correction Line Trail,12,2,1,2,['Gogebic' 'Ontonagon'],['Porcupine Mountains Wilderness State Park']


## 4. Final segment-level GeoDataFrame

`TrailGroupName` combines county and hiking name to distinguish similarly named trails in different locations. The data remains at the segment level so geometry can be reviewed before any dissolve operation.

In [22]:
DROP_COLUMNS = [
    "Peninsula",
    "Hiking",
    "TrailApprovalStatus",
    "SpecialRestrictionType",
    "RecreationSearchFacilityID",
    "RecreationSearchTrailID",
    "last_edited_date",
]

final_trails = audit_trails.drop(
    columns=DROP_COLUMNS
).copy()

final_trails["TrailGroupName"] = (
    final_trails["County"]
    + " | "
    + final_trails["HikingName"]
)

FINAL_COLUMN_ORDER = [
    "OBJECTID",
    "TrailGroupName",
    "HikingName",
    "TrailNamePrimary",
    "FacilityName",
    "County",
    "OpenClosedStatusNonmotor",
    "SurfaceType",
    "ADAAccessible",
    "SegmentLengthMiles",
    "TrailAdministrator",
    "geometry",
]

final_trails = final_trails[FINAL_COLUMN_ORDER]

final_trails.head()

,OBJECTID,TrailGroupName,HikingName,TrailNamePrimary,FacilityName,County,OpenClosedStatusNonmotor,SurfaceType,ADAAccessible,SegmentLengthMiles,TrailAdministrator,geometry
0,1919,Mackinac | Straits Main Trail,Straits Main Trail,Straits - Main Trail,Straits State Park,Mackinac,Open,Dirt Natural,Not Accessible,0.006157,DNR Parks And Recreation PRD,"LINESTRING (-84.72138 45.84982, -84.72135 45.8..."
1,2486,Marquette | Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Temporarily Closed,Dirt Natural,Not Accessible,1.200800,DNR Parks And Recreation PRD,"LINESTRING (-87.47649 46.63472, -87.47644 46.6..."
2,2495,Mackinac | North Country Trail,North Country Trail,UP 2,<NA>,Mackinac,Open,Dirt Natural,Not Accessible,3.891672,DNR Parks And Recreation PRD,"LINESTRING (-84.73683 45.87386, -84.73687 45.8..."
3,2540,Schoolcraft | Gemini Lake Pathway,Gemini Lake Pathway,Gemini Lake Pathway,State Forest,Schoolcraft,Open,Dirt Natural,Not Accessible,0.514227,DNR Parks And Recreation PRD,"LINESTRING (-86.30655 46.48498, -86.30623 46.4..."
4,2541,Schoolcraft | Gemini Lake Pathway,Gemini Lake Pathway,Gemini Lake Pathway,State Forest,Schoolcraft,Open,Dirt Natural,Not Accessible,0.264824,DNR Parks And Recreation PRD,"LINESTRING (-86.30464 46.48159, -86.30522 46.4..."


In [23]:
# Validate row preservation, identifier uniqueness, and grouping coverage.
assert len(final_trails) == len(trails)
assert final_trails["OBJECTID"].is_unique
assert final_trails["TrailGroupName"].notna().all()
assert set(DROP_COLUMNS).isdisjoint(final_trails.columns)

final_summary = pd.Series(
    {
        "segment_rows": len(final_trails),
        "trail_groups": final_trails["TrailGroupName"].nunique(),
        "counties": final_trails["County"].nunique(),
        "missing_geometry": int(final_trails.geometry.isna().sum()),
        "invalid_geometry": int((~final_trails.geometry.is_valid).sum()),
    },
    name="value",
)

final_summary

segment_rows        2439
trail_groups         159
counties              15
missing_geometry       0
invalid_geometry       0
Name: value, dtype: int64

In [24]:
trail_group_summary = (
    final_trails.groupby("TrailGroupName")
    .agg(
        segment_count=("OBJECTID", "size"),
        reported_length_miles=("SegmentLengthMiles", "sum"),
        facility_count=("FacilityName", "nunique"),
        surface_count=("SurfaceType", "nunique"),
        status_count=("OpenClosedStatusNonmotor", "nunique"),
    )
    .sort_values("reported_length_miles", ascending=False)
)

trail_group_summary.head(30)

,segment_count,reported_length_miles,facility_count,surface_count,status_count
TrailGroupName,,,,,
Alger | North Country Trail,129,95.559970,1,2,1
Marquette | North Country Trail,60,74.256774,1,4,1
Ontonagon | North Country Trail,100,73.306963,1,3,1
Chippewa | North Country Trail,148,66.307730,1,6,1
Mackinac | Iron Belle Trail,1,63.804200,0,1,1
Delta | Iron Belle Trail,6,52.355545,0,1,1
Marquette | Iron Ore Heritage Trail,49,50.502948,0,4,1
Baraga | North Country Trail,21,47.588697,1,2,1
Gogebic | North Country Trail,24,43.997004,1,3,1


In [25]:
# All source segments should appear in exactly one trail group.
assert trail_group_summary["segment_count"].sum() == len(final_trails)

# Each County | HikingName group should be unique.
assert trail_group_summary.index.is_unique

# Group names should be complete.
assert final_trails["TrailGroupName"].notna().all()

print("Trail grouping validation passed.")

Trail grouping validation passed.


## 5. Save the cleaned segment data

In [26]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

final_trails.to_parquet(
    PROCESSED_PATH,
    index=False,
)

print(f"Saved {len(final_trails):,} rows to {PROCESSED_PATH}")

Saved 2,439 rows to ..\data\processed\dnr_up_hiking_segments_clean.parquet
